# English corpus cleaning

This notebook runs the complete cleaning pipeline from `preprocess_corpora.py` on the English raw dataset and writes the official English corpus.

In [1]:
# Install every dependency required by the preprocessing pipeline.
%pip install beautifulsoup4 ftfy regex phonenumbers emoji datasketch spylls sentence-transformers torch numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import importlib.util
import sys

# Locate the shared preprocessing module whether Jupyter starts in the project root or in this folder.
candidate_paths = (
    Path.cwd() / "preprocess_corpora.py",
    Path.cwd() / "3-cleaning" / "preprocess_corpora.py",
)
script_path = next((path.resolve() for path in candidate_paths if path.is_file()), None)
if script_path is None:
    raise FileNotFoundError("Could not locate 3-cleaning/preprocess_corpora.py")

spec = importlib.util.spec_from_file_location("preprocess_corpora", script_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load preprocessing module from {script_path}")

preprocess_corpora = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = preprocess_corpora
spec.loader.exec_module(preprocess_corpora)

LANGUAGE = "EN"
input_path = preprocess_corpora.INPUT_PATHS[LANGUAGE]
if not input_path.is_file():
    raise FileNotFoundError(f"English raw dataset not found: {input_path}")

print(f"Input dataset: {input_path}")
print(f"Output dataset: {preprocess_corpora.OUTPUT_PATHS[LANGUAGE]}")

Input dataset: /home/user/J2C-TST/2-raw_data/EN/raw_dataset_EN.csv
Output dataset: /home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv


In [3]:
# Run normalization, quality filtering, deduplication, and dataset splitting.
output_path = preprocess_corpora.process_corpus(LANGUAGE)

Input: 4963 rows
Removed because of empty text: 0
Removed because of conversational_hashtags: 7
Removed because of conversational_character_count: 2110
Removed because of conversational_emoji: 1
Removed because of journalistic_style:duplicate_9grams: 1
Removed because of journalistic_word_count: 12
Removed because of conversational_style:alphabetic_words: 1
Removed because of embedding similarity: 3
Removed as exact duplicates: 1437
Removed as near-duplicates: 1
Removed from training because of test overlap: 0
Output: 1390 rows in /home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv


In [4]:
import csv
from collections import Counter

# Confirm that the generated corpus has the expected text and split columns.
with output_path.open("r", encoding="utf-8", newline="") as source:
    reader = csv.DictReader(source)
    rows = list(reader)
    fieldnames = reader.fieldnames or []

required_columns = {"journalistic_style", "conversational_style", "split"}
missing_columns = required_columns.difference(fieldnames)
if missing_columns:
    raise ValueError(f"Missing output columns: {sorted(missing_columns)}")

split_counts = Counter(row["split"] for row in rows)
print(f"Generated {len(rows)} rows at {output_path}")
print(f"Split counts: {dict(split_counts)}")

Generated 1390 rows at /home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv
Split counts: {'train': 973, 'test': 278, 'validation': 139}
